# Phase 1: Exploratory Data Analysis & Pipeline Preprocessing
### Customer Churn Decision Engine
**Objective:** Audit the IBM Telco Customer Churn dataset, uncover key churn drivers, diagnose data hygiene anomalies, and validate the custom Scikit-Learn preprocessing pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style="whitegrid")

# Load raw dataset
DATA_PATH = Path("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = pd.read_csv(DATA_PATH)
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 1. Data Hygiene & Anomaly Audit

In [ ]:
# Detect whitespace strings in TotalCharges
blank_mask = df["TotalCharges"].astype(str).str.strip() == ""
blank_count = blank_mask.sum()
print(f"Rows with blank TotalCharges: {blank_count}")
print("Tenure distribution for blank TotalCharges rows:")
print(df.loc[blank_mask, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]])

## 2. Target Distribution & Class Imbalance

In [ ]:
churn_counts = df["Churn"].value_counts(normalize=True) * 100
print("Churn Distribution (%):")
print(churn_counts.round(2))

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x="Churn", palette=["#2ecc71", "#e74c3c"], ax=ax)
ax.set_title("Customer Churn Class Balance (26.5% Positive Churn Rate)", fontsize=12, fontweight="bold")
ax.set_ylabel("Customer Count")
plt.tight_layout()
plt.show()

## 3. High-Impact Drivers: Contract, Payment & Internet Service

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x="Contract", hue="Churn", palette="Set2", ax=axes[0])
axes[0].set_title("Churn Rate by Contract Term", fontweight="bold")
axes[0].tick_params(axis='x', rotation=15)

sns.countplot(data=df, x="InternetService", hue="Churn", palette="Set2", ax=axes[1])
axes[1].set_title("Churn Rate by Internet Service", fontweight="bold")

sns.countplot(data=df, x="PaymentMethod", hue="Churn", palette="Set2", ax=axes[2])
axes[2].set_title("Churn Rate by Payment Method", fontweight="bold")
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. Pipeline Integration & Transformation Validation

In [ ]:
import sys
sys.path.append("..")
from src.data.preprocessor import create_preprocessor_pipeline, load_raw_dataset, split_data, get_feature_names

X, y = load_raw_dataset(DATA_PATH)
X_train, X_test, y_train, y_test = split_data(X, y)

pipeline = create_preprocessor_pipeline(drop_id=True)
X_train_transformed = pipeline.fit_transform(X_train)
X_test_transformed = pipeline.transform(X_test)

print(f"Transformed Training Matrix Shape: {X_train_transformed.shape}")
print(f"Transformed Testing Matrix Shape:  {X_test_transformed.shape}")
print("Sample Feature Names:")
feature_names = get_feature_names(pipeline.named_steps["preprocessor"])
print(feature_names[:10])